::: {.sb-downloads}
[Download this notebook](../downloads/solutions-911-emergencies.ipynb){download="solutions-911-emergencies.ipynb"}  
Data: [911_calls_2019.csv](../downloads/data/911_calls_2019.csv){download="911_calls_2019.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Back to the exercises](../exercises/911-emergencies.ipynb)
:::

::: {.callout-tip}
Try the exercises on your own first: you learn much more by searching than by reading a solution.
:::

Montgomery County, in Pennsylvania (USA), wants to know where and when its emergency services are needed most, to place ambulances and plan road safety work. It gives you a sample of the 911 calls it received in 2019. Your job: explore the calls, find the **hot spots** with DBSCAN, and make recommendations.

## The dataset 📞 {#ex-the-dataset}
`../data/911_calls_2019.csv` holds 20,000 emergency calls of 2019 (a random sample of the calls published by the county). Each row is one call:

* `time`: date and time of the call;
* `lat`, `lng`: latitude and longitude of the emergency, in degrees;
* `category`: `EMS` (medical emergency), `Fire` or `Traffic`;
* `reason`: the reason of the call, for example `CARDIAC EMERGENCY` or `VEHICLE ACCIDENT`;
* `township`: the township (town) of the call.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

EARTH_RADIUS_M = 6_371_000   # radius of the Earth, in metres
pd.set_option("display.max_colwidth", 100)

## Explore the calls 🔍 {#ex-explore-the-calls}

1. Load the dataset into a DataFrame called `calls`, with the `time` column read as dates. Display the first rows and the number of calls.

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `pd.read_csv(..., parse_dates=["time"])`.
:::

In [ ]:
calls = pd.read_csv("../data/911_calls_2019.csv", parse_dates=["time"])
print(calls.shape)
calls.head()

(20000, 6)


,time,lat,lng,category,reason,township
0,2019-01-01 00:02:30,40.25048,-75.40729,EMS,OVERDOSE,LOWER SALFORD
1,2019-01-01 01:05:56,40.01868,-75.27390,EMS,CARDIAC EMERGENCY,LOWER MERION
2,2019-01-01 01:28:54,40.30263,-75.29130,Traffic,HAZARDOUS ROAD CONDITIONS,BUCKS COUNTY
3,2019-01-01 02:03:49,40.08799,-75.37794,EMS,CHOKING,UPPER MERION
4,2019-01-01 02:29:29,40.24709,-75.65328,EMS,FALL VICTIM,POTTSTOWN


2. How many calls are there in each category? Show it as a bar chart.

In [ ]:
per_category = calls["category"].value_counts()
print((per_category / len(calls)).round(3))
px.bar(per_category, title="Number of calls per category").update_layout(showlegend=False)

category
EMS        0.502
Traffic    0.349
Fire       0.150
Name: count, dtype: float64


**Half of the calls are medical emergencies (EMS, 10,032 calls), 35% are traffic incidents (6,978) and 15% are fires (2,990).**

3. Display the 10 most frequent reasons, with their category, in a horizontal bar chart. What do you notice?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `calls.groupby(["category", "reason"]).size()`, then `.nlargest(10)` and `.reset_index(name="calls")`.
:::

In [ ]:
top_reasons = calls.groupby(["category", "reason"]).size().nlargest(10).reset_index(name="calls")
fig = px.bar(top_reasons, x="calls", y="reason", color="category", orientation="h",
             title="The 10 most frequent reasons of a call")
fig.update_yaxes(categoryorder="total ascending")
fig.show()

**Vehicle accidents are by far the first reason: 4,456 traffic calls, three times more than the second reason, disabled vehicles (1,443). They also appear in the medical calls (777 EMS calls for people injured in an accident). Then come fire alarms (1,142) and three medical reasons with about 1,000 calls each: falls, respiratory emergencies and cardiac emergencies.**

4. Count the calls per hour of the day and per category, and draw them as lines. Do the same per day of the week. What do you notice?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `calls["time"].dt.hour` and `calls["time"].dt.day_name()`. To keep the days in order, pass `category_orders={"day": ["Monday", ..., "Sunday"]}` to plotly.
:::

In [ ]:
calls["hour"] = calls["time"].dt.hour
calls["day"] = calls["time"].dt.day_name()

per_hour = calls.groupby(["hour", "category"]).size().reset_index(name="calls")
px.line(per_hour, x="hour", y="calls", color="category", markers=True, title="Calls per hour of the day").show()

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
per_day = calls.groupby(["day", "category"]).size().reset_index(name="calls")
px.line(per_day, x="day", y="calls", color="category", markers=True, category_orders={"day": days},
        title="Calls per day of the week").show()

**There are far more calls during the day than at night, with a minimum around 3–4 am. Traffic calls follow the commute: a first peak at 8–9 am and the highest one at 5 pm (625 calls), and far fewer calls at the weekend (865 on Saturdays, 622 on Sundays, against 1,050 to 1,150 on weekdays). Medical calls are much more regular: from 1,321 (Sunday) to 1,507 (Tuesday) per day of the week, with a long plateau from 9 am to 6 pm. Emergencies happen every day.**

5. Draw all the calls on a map, with one colour per category. What do you see?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `px.scatter_map(calls, lat="lat", lon="lng", color="category", map_style="carto-positron", zoom=9, opacity=0.4)`.
:::

In [ ]:
fig = px.scatter_map(calls, lat="lat", lon="lng", color="category", opacity=0.4, zoom=9, height=550,
                     map_style="carto-positron", title="The 20,000 calls of 2019")
fig.update_traces(marker_size=4)
fig.show()

**The calls cover the whole county, but they are much denser in the south-east, close to Philadelphia, and in the towns (Norristown, Pottstown, Lansdale...). Traffic calls draw the main roads: we can follow the highways on the map. It's hard to see more with 20,000 points: we need DBSCAN to find the places where calls really concentrate.**

## Distances on a map 🌍 {#ex-distances-on-a-map}

6. Latitude and longitude are angles in degrees. One degree of latitude is about 111.2 km everywhere; one degree of longitude is about $111.2 \times \cos(\text{latitude})$ km. Compute how many metres 0.01° of latitude and 0.01° of longitude represent at the mean latitude of the calls. Why is a Euclidean distance on (`lat`, `lng`) a bad choice? What should you use instead?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `np.cos` needs radians: `np.cos(np.radians(latitude))`.
:::

In [ ]:
mean_lat = calls["lat"].mean()
print(f"mean latitude: {mean_lat:.2f}°")
print(f"0.01° of latitude : {0.01 * 111_200:.0f} m")
print(f"0.01° of longitude: {0.01 * 111_200 * np.cos(np.radians(mean_lat)):.0f} m")

mean latitude: 40.16°
0.01° of latitude : 1112 m
0.01° of longitude: 850 m


**At this latitude (40.16° N), 0.01° of latitude is 1,112 m but 0.01° of longitude is only 850 m. A Euclidean distance on degrees mixes two different units: its "circle" is an ellipse on the ground, and its `eps` has no meaning in metres. We use the haversine distance (`metric="haversine"`), with the coordinates in radians and `eps` in metres divided by the radius of the Earth.**

## Hot spots of medical emergencies 🚑 {#ex-hot-spots-of-medical-emergencies}

7. Keep only the `EMS` calls in a DataFrame called `ems`. Build `X_ems`, the array of their `[lat, lng]` coordinates **in radians**. How many EMS calls are there, and how many distinct locations?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `np.radians(ems[["lat", "lng"]].to_numpy())`; `drop_duplicates()` counts the distinct locations.
:::

In [ ]:
ems = calls[calls["category"] == "EMS"].copy()
X_ems = np.radians(ems[["lat", "lng"]].to_numpy())
print(f"{len(ems)} EMS calls at {len(ems[['lat', 'lng']].drop_duplicates())} distinct locations")
ems.groupby(["lat", "lng", "township"]).size().nlargest(3)

10032 EMS calls at 4423 distinct locations


lat       lng        township        
40.13304  -75.40846  LOWER PROVIDENCE    231
40.28903  -75.39959  LOWER SALFORD        79
40.10740  -75.11321  ABINGTON             50
dtype: int64

**10,032 EMS calls, but only 4,423 distinct locations: the position of a call is the address of the emergency, and some addresses call again and again. The busiest one, in Lower Providence, received 231 calls in the year (about 1 call every 1.6 days).**

8. The county defines a hot spot as **at least 20 calls in the year in a small area** (about one call every two or three weeks), so we take `min_samples=20`. Draw the k-distance plot with `k = 20`: for every call, the distance **in metres** to its 20th nearest call (itself included), sorted. Where is the elbow?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `NearestNeighbors(n_neighbors=20, metric="haversine").fit(X_ems).kneighbors(X_ems)` returns the distances (in radians) and the indices. Take the last column of the distances, multiply by `EARTH_RADIUS_M`, sort with `np.sort`. Limit the y axis to 0–3,000 m to see the elbow.
:::

In [ ]:
k = 20
distances, _ = NearestNeighbors(n_neighbors=k, metric="haversine").fit(X_ems).kneighbors(X_ems)
k_distance = np.sort(distances[:, -1]) * EARTH_RADIUS_M

fig = px.line(y=k_distance, title="k-distance plot of the EMS calls (k = 20)",
              labels={"x": "calls, sorted by their k-distance", "y": "distance to the 20th call (m)"})
fig.update_yaxes(range=[0, 3000])
fig.show()
print(f"calls with a k-distance of 0 m: {(k_distance == 0).sum()}")
print("k-distance at 50%, 80%, 90%, 95% of the calls:", np.percentile(k_distance, [50, 80, 90, 95]).round(-1))

calls with a k-distance of 0 m: 1748
k-distance at 50%, 80%, 90%, 95% of the calls: [ 500.  930. 1260. 1580.]


**The curve starts at 0 m (1,748 calls have at least 19 other calls at exactly the same address), then rises slowly: half of the calls have 20 calls within 500 m. It bends upwards around 1,000–1,500 m (80% of the calls are below 930 m, 95% below 1,580 m): the elbow is around 1 km.**

9. Fit a DBSCAN with `eps` = 1,000 m (the elbow), `min_samples=20` and the haversine distance. How many clusters do you get, and what share of the calls is noise? How many calls and how many townships does the largest cluster contain? Is it a useful "hot spot"?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `DBSCAN(eps=1000 / EARTH_RADIUS_M, min_samples=20, metric="haversine", algorithm="ball_tree")`. Use `fit_predict` to get the labels, and `value_counts()` to find the largest cluster.
:::

In [ ]:
labels_1km = DBSCAN(eps=1000 / EARTH_RADIUS_M, min_samples=20, metric="haversine",
                    algorithm="ball_tree").fit_predict(X_ems)
sizes = pd.Series(labels_1km[labels_1km != -1]).value_counts()
largest = sizes.index[0]
print(f"{len(sizes)} clusters, {np.mean(labels_1km == -1):.1%} of noise")
print(f"largest cluster: {sizes.iloc[0]} calls in {ems['township'][labels_1km == largest].nunique()} townships")

27 clusters, 8.3% of noise
largest cluster: 2889 calls in 21 townships


**27 clusters and only 8.3% of noise. But the largest cluster holds 2,889 calls spread over 21 townships: with a radius of 1 km, the dense neighbourhoods touch each other and DBSCAN chains whole towns together. The elbow gives the radius where the county stops being "dense", not the size of a hot spot. To place an ambulance or a prevention action, we need much smaller areas.**

10. The county wants hot spots of a few streets: fit DBSCAN again with `eps` = 300 m (and still `min_samples=20`). Store the labels in a new column `cluster` of `ems`. How many hot spots do you find, and what share of the calls is noise?

In [ ]:
ems_db = DBSCAN(eps=300 / EARTH_RADIUS_M, min_samples=20, metric="haversine", algorithm="ball_tree")
ems["cluster"] = ems_db.fit_predict(X_ems)
n_hot_spots = ems["cluster"].max() + 1
print(f"{n_hot_spots} hot spots, {np.mean(ems['cluster'] == -1):.1%} of the EMS calls are noise")
print(f"calls in a hot spot: {(ems['cluster'] != -1).sum()}")

91 hot spots, 59.4% of the EMS calls are noise
calls in a hot spot: 4069


**91 hot spots, and 59.4% of the calls are noise. That's expected and it's fine: most medical emergencies happen at home, scattered all over the county. The 4,069 calls in a hot spot are the places where emergencies repeat.**

11. Draw the calls that belong to a hot spot on a map (leave the noise out), with one colour per cluster.

::: {.callout-tip collapse="true" title="Show hint"}
Hint: filter with `ems[ems["cluster"] != -1]`; convert the cluster number to text (`.astype(str)`) so that plotly uses distinct colours instead of a colour scale.
:::

In [ ]:
hot = ems[ems["cluster"] != -1]
fig = px.scatter_map(hot, lat="lat", lon="lng", color=hot["cluster"].astype(str), zoom=9, height=550,
                     hover_data=["township", "reason"], map_style="carto-positron",
                     title="EMS calls in a hot spot (noise left out)")
fig.update_layout(showlegend=False)
fig.show()

12. Describe the hot spots: for each cluster, compute the number of calls, the centre (mean latitude and longitude), the main township, the number of townships and the 3 most frequent reasons. Display the 8 largest. Then draw one bubble per hot spot on a map, sized by the number of calls. What do you notice?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `ems[ems["cluster"] != -1].groupby("cluster").agg(calls=("reason", "size"), lat=("lat", "mean"), ...)`. For the main township and the top reasons, pass a function, for example `lambda r: ", ".join(r.value_counts().index[:3])`.
:::

In [ ]:
def describe_hot_spots(df):
    # One row per cluster (noise excluded), largest first
    return (df[df["cluster"] != -1].groupby("cluster")
            .agg(calls=("reason", "size"), lat=("lat", "mean"), lng=("lng", "mean"),
                 township=("township", lambda t: t.value_counts().index[0]),
                 n_townships=("township", "nunique"),
                 top_reasons=("reason", lambda r: ", ".join(r.value_counts().index[:3])))
            .sort_values("calls", ascending=False))

ems_spots = describe_hot_spots(ems)
ems_spots.head(8)

,calls,lat,lng,township,n_townships,top_reasons
cluster,,,,,,
1,624,40.119285,-75.343215,NORRISTOWN,6,"RESPIRATORY EMERGENCY, CARDIAC EMERGENCY, SUBJECT IN PAIN"
0,301,40.247162,-75.645877,POTTSTOWN,2,"RESPIRATORY EMERGENCY, CARDIAC EMERGENCY, SUBJECT IN PAIN"
3,231,40.133040,-75.408460,LOWER PROVIDENCE,1,"MEDICAL ALERT ALARM, FALL VICTIM, HEAD INJURY"
43,97,40.245277,-75.283728,LANSDALE,2,"SUBJECT IN PAIN, CARDIAC EMERGENCY, RESPIRATORY EMERGENCY"
13,80,40.229010,-75.387850,CHESTER COUNTY,7,"VEHICLE ACCIDENT, RESPIRATORY EMERGENCY, FALL VICTIM"
39,79,40.289030,-75.399590,LOWER SALFORD,1,"FALL VICTIM, HEAD INJURY, CARDIAC EMERGENCY"
20,76,40.104141,-75.341085,BRIDGEPORT,2,"SUBJECT IN PAIN, CARDIAC EMERGENCY, DIABETIC EMERGENCY"
56,65,40.008665,-75.292437,LOWER MERION,1,"FALL VICTIM, RESPIRATORY EMERGENCY, SUBJECT IN PAIN"


In [ ]:
fig = px.scatter_map(ems_spots.reset_index(), lat="lat", lon="lng", size="calls", size_max=35, zoom=9, height=550,
                     hover_data=["township", "top_reasons"], map_style="carto-positron",
                     title="The 91 hot spots of medical emergencies (bubble size = number of calls)")
fig.show()

**The two largest hot spots are the centres of the two main towns: Norristown (624 calls) and Pottstown (301 calls), with the most common medical reasons (respiratory and cardiac emergencies, people in pain). The third one is very different: 231 calls at a single address in Lower Providence, mostly medical alert alarms, falls and head injuries. It looks like a home for elderly people: one place where a prevention action (against falls, for instance) could avoid many calls. The fifth one is a warning about the data: 80 calls on a single point, with reasons that have nothing in common (vehicle accidents, breathing problems, falls...), and townships that are mostly other counties (`CHESTER COUNTY`, `BUCKS COUNTY`, `BERKS COUNTY`...). It is certainly a default position given to calls located outside the county, not a real hot spot: such points must be cleaned before drawing conclusions.**

## Hot spots of traffic incidents 🚗 {#ex-hot-spots-of-traffic-incidents}

13. Do the same for the `Traffic` calls: keep them in `traffic`, fit DBSCAN with the same definition of a hot spot (300 m, `min_samples=20`), count the hot spots and the noise, map the calls in a hot spot, and describe the 8 largest hot spots. How do they differ from the EMS hot spots?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: reuse the code of questions 7, 10, 11 and 12; the function `describe_hot_spots` works on any DataFrame with a `cluster` column.
:::

In [ ]:
traffic = calls[calls["category"] == "Traffic"].copy()
X_traffic = np.radians(traffic[["lat", "lng"]].to_numpy())
traffic["cluster"] = DBSCAN(eps=300 / EARTH_RADIUS_M, min_samples=20, metric="haversine",
                            algorithm="ball_tree").fit_predict(X_traffic)
print(f"{len(traffic)} traffic calls: {traffic['cluster'].max() + 1} hot spots, "
      f"{np.mean(traffic['cluster'] == -1):.1%} of noise")

hot = traffic[traffic["cluster"] != -1]
fig = px.scatter_map(hot, lat="lat", lon="lng", color=hot["cluster"].astype(str), zoom=9, height=550,
                     hover_data=["township", "reason"], map_style="carto-positron",
                     title="Traffic calls in a hot spot (noise left out)")
fig.update_layout(showlegend=False)
fig.show()

traffic_spots = describe_hot_spots(traffic)
traffic_spots.head(8)

6978 traffic calls: 39 hot spots, 74.9% of noise


,calls,lat,lng,township,n_townships,top_reasons
cluster,,,,,,
6,256,40.097220,-75.376200,UPPER MERION,1,"VEHICLE ACCIDENT, DISABLED VEHICLE, ROAD OBSTRUCTION"
7,94,40.069379,-75.316691,WEST CONSHOHOCKEN,1,"VEHICLE ACCIDENT, DISABLED VEHICLE, ROAD OBSTRUCTION"
4,82,40.108258,-75.306226,PLYMOUTH,1,"VEHICLE ACCIDENT, DISABLED VEHICLE, ROAD OBSTRUCTION"
11,80,40.024951,-75.282936,LOWER MERION,1,"DISABLED VEHICLE, VEHICLE ACCIDENT, ROAD OBSTRUCTION"
20,74,40.115038,-75.343993,NORRISTOWN,3,"VEHICLE ACCIDENT, DISABLED VEHICLE, ROAD OBSTRUCTION"
3,70,40.153260,-75.189513,UPPER DUBLIN,1,"VEHICLE ACCIDENT, ROAD OBSTRUCTION, DISABLED VEHICLE"
2,67,40.229010,-75.387850,CHESTER COUNTY,5,"VEHICLE ACCIDENT, DISABLED VEHICLE, ROAD OBSTRUCTION"
18,64,40.008571,-75.291660,LOWER MERION,1,"VEHICLE ACCIDENT, DISABLED VEHICLE, ROAD OBSTRUCTION"


**39 traffic hot spots, with 74.9% of noise: accidents are spread along thousands of kilometres of roads. The hot spots are on the main roads and their junctions, and almost all of them have the same top reasons: vehicle accidents, disabled vehicles, road obstructions. The largest one, in Upper Merion, gathers 256 calls at a single point, more than twice the next one (94 calls in West Conshohocken): probably a major highway junction. The default point of the calls from other counties appears again (67 calls, "CHESTER COUNTY"). A few hot spots are stretched along a road, up to about 1 km long, rather than round: DBSCAN follows such shapes naturally.**

## Recommendations 📋 {#ex-recommendations}

14. Using everything you found, what would you recommend to the county? Write 3 or 4 recommendations.

To go further: find the locations of the ambulance and fire stations of the county (they are public) and compare them with the hot spots. Are some hot spots far from any station?

**Recommendations:**

* **Staff by the clock:** most calls arrive between 8 am and 8 pm, and traffic calls peak at the evening rush hour on weekdays. Plan more crews at these times, fewer at night.
* **Ambulances near the urban hot spots:** Norristown and Pottstown concentrate the largest clusters of medical emergencies. Ambulances waiting near these centres reach many calls quickly.
* **Prevention at repeat addresses:** some hot spots are single addresses with hundreds of calls (falls and medical alarms at what looks like a home for elderly people). A prevention plan with these places (fall prevention, medical staff on site) can avoid many calls.
* **Road safety at the traffic hot spots:** the largest traffic hot spot (Upper Merion) and the other junctions found by DBSCAN are where changes of road layout, speed limits or signs would prevent the most accidents.
* **Check the data first:** one point gathers the calls located in other counties (80 EMS and 67 traffic calls in our sample). Remove such default positions before taking decisions from the map.